# RAG Fundamentals: Building Your First RAG Pipeline

## Learning Objectives
- Understand RAG architecture and use cases
- Learn different chunking strategies
- Use Gemini embeddings with ChromaDB
- Build a basic RAG pipeline

## Prerequisites
- Python 3.11+
- Gemini API key
- Basic understanding of NLP concepts

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure Gemini API
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

print("Gemini API configured successfully!")
print(f"Available models: {[m.name for m in genai.list_models() if 'embedding' in m.name.lower() or 'pro' in m.name.lower()][:5]}")

## 1. What is RAG?

**Retrieval-Augmented Generation (RAG)** combines:
1. **Retrieval**: Finding relevant information from a knowledge base
2. **Generation**: Using that information to generate accurate answers

### Why RAG?
- Reduces hallucinations
- Provides up-to-date information
- Enables domain-specific knowledge
- More transparent than pure LLM responses

## 2. Document Processing & Chunking

Let's explore different ways to split documents into chunks for retrieval.

In [ ]:
# Load sample documents
def load_sample_documents():
    """Load sample text files from data directory."""
    documents = []
    data_dir = "data/sample_texts"
    
    for filename in os.listdir(data_dir):
        if filename.endswith('.txt'):
            filepath = os.path.join(data_dir, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                text = f.read()
                documents.append({
                    'text': text,
                    'metadata': {'source': filename}
                })
    
    return documents

docs = load_sample_documents()
print(f"Loaded {len(docs)} documents")
for doc in docs:
    print(f"- {doc['metadata']['source']}: {len(doc['text'])} characters")

In [ ]:
from utils.chunking import chunk_text, get_chunk_stats

# Demonstrate different chunking methods
sample_text = docs[0]['text']

print("=== Chunking Methods Comparison ===\n")

# Fixed-size chunking
fixed_chunks = chunk_text(sample_text, method="fixed", chunk_size=200, overlap=50)
print(f"Fixed-size chunks: {len(fixed_chunks)} chunks")
print(f"  Average length: {sum(len(c) for c in fixed_chunks) / len(fixed_chunks):.0f} chars")
print(f"  First chunk preview: {fixed_chunks[0][:100]}...\n")

# Recursive chunking
recursive_chunks = chunk_text(sample_text, method="recursive", chunk_size=200)
print(f"Recursive chunks: {len(recursive_chunks)} chunks")
print(f"  Average length: {sum(len(c) for c in recursive_chunks) / len(recursive_chunks):.0f} chars")
print(f"  First chunk preview: {recursive_chunks[0][:100]}...\n")

# Semantic chunking
semantic_chunks = chunk_text(sample_text, method="semantic", chunk_size=200)
print(f"Semantic chunks: {len(semantic_chunks)} chunks")
print(f"  Average length: {sum(len(c) for c in semantic_chunks) / len(semantic_chunks):.0f} chars")
print(f"  First chunk preview: {semantic_chunks[0][:100]}...")

### Chunking Strategy Trade-offs

| Method | Pros | Cons | Best For |
|--------|------|------|----------|
| **Fixed-size** | Simple, predictable | May split sentences | Quick prototyping |
| **Recursive** | Respects document structure | May have uneven sizes | Well-structured docs |
| **Semantic** | Groups related content | More complex | Knowledge bases |

In [ ]:
# Visualize chunk statistics
import matplotlib.pyplot as plt

methods = ['fixed', 'recursive', 'semantic']
chunk_counts = []
avg_lengths = []

for method in methods:
    chunks = chunk_text(sample_text, method=method, chunk_size=200, overlap=50)
    stats = get_chunk_stats([{'text': c, 'metadata': {}} for c in chunks])
    chunk_counts.append(stats['count'])
    avg_lengths.append(stats['avg_length'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(methods, chunk_counts, color=['blue', 'green', 'orange'])
ax1.set_title('Number of Chunks by Method')
ax1.set_ylabel('Chunk Count')

ax2.bar(methods, avg_lengths, color=['blue', 'green', 'orange'])
ax2.set_title('Average Chunk Length by Method')
ax2.set_ylabel('Characters')

plt.tight_layout()
plt.show()

## 3. Gemini Embeddings

Gemini provides powerful text embeddings that capture semantic meaning.

In [ ]:
# Test Gemini embedding
test_text = "Machine learning is a subset of artificial intelligence"

# Get embedding
embedding_result = genai.embed_content(
    model="models/text-embedding-004",
    content=test_text,
    task_type="retrieval_document"
)

embedding = embedding_result['embedding']
print(f"Text: {test_text}")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")
print(f"Embedding norm: {sum(x**2 for x in embedding)**0.5:.4f}")

In [ ]:
# Compare embeddings for similar vs different texts
texts = [
    "Machine learning algorithms",
    "Deep learning neural networks",
    "Natural language processing",
    "The weather is sunny today"
]

embeddings = []
for text in texts:
    result = genai.embed_content(
        model="models/text-embedding-004",
        content=text,
        task_type="retrieval_document"
    )
    embeddings.append(result['embedding'])

# Calculate similarity matrix
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similarity Matrix:")
print("=" * 50)
for i, text1 in enumerate(texts):
    for j, text2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"{text1[:20]:20} vs {text2[:20]:20}: {sim:.3f}")
    print()

## 4. Vector Database with ChromaDB

ChromaDB stores embeddings and enables fast similarity search.

In [ ]:
import chromadb

# Create ChromaDB client and collection
client = chromadb.Client()
collection = client.create_collection(name="rag_fundamentals")

print("ChromaDB collection created!")

In [ ]:
# Prepare documents for storage
all_chunks = []
all_metadatas = []
all_ids = []

for doc_idx, doc in enumerate(docs):
    # Chunk the document
    chunks = chunk_text(doc['text'], method="recursive", chunk_size=200)
    
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({
            'source': doc['metadata']['source'],
            'chunk_index': chunk_idx,
            'doc_index': doc_idx
        })
        all_ids.append(f"doc{doc_idx}_chunk{chunk_idx}")

print(f"Total chunks to store: {len(all_chunks)}")

# Generate embeddings for all chunks
print("Generating embeddings...")
chunk_embeddings = []
for chunk in all_chunks:
    result = genai.embed_content(
        model="models/text-embedding-004",
        content=chunk,
        task_type="retrieval_document"
    )
    chunk_embeddings.append(result['embedding'])

print("Embeddings generated!")

In [ ]:
# Add documents to ChromaDB
collection.add(
    documents=all_chunks,
    embeddings=chunk_embeddings,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"Added {collection.count()} chunks to ChromaDB")

## 5. Basic RAG Pipeline

Now let's build the complete RAG pipeline: Query → Embed → Retrieve → Generate

In [ ]:
def rag_query(query, collection, top_k=3):
    """
    Complete RAG pipeline.
    
    Args:
        query: User question
        collection: ChromaDB collection
        top_k: Number of chunks to retrieve
    
    Returns:
        Dictionary with retrieved chunks and generated answer
    """
    # Step 1: Embed the query
    query_embedding = genai.embed_content(
        model="models/text-embedding-004",
        content=query,
        task_type="retrieval_query"
    )['embedding']
    
    # Step 2: Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    retrieved_chunks = results['documents'][0]
    retrieved_metadatas = results['metadatas'][0]
    distances = results['distances'][0]
    
    # Step 3: Create context from retrieved chunks
    context = "\n\n".join(retrieved_chunks)
    
    # Step 4: Generate answer using Gemini
    prompt = f"""Answer the following question based on the provided context. 
    If the context doesn't contain enough information, say so.
    
    Context:
    {context}
    
    Question: {query}
    
    Answer:"""
    
    model = genai.GenerativeModel('gemini-1.5-pro')
    response = model.generate_content(prompt)
    
    return {
        'query': query,
        'answer': response.text,
        'retrieved_chunks': retrieved_chunks,
        'retrieved_metadatas': retrieved_metadatas,
        'distances': distances
    }

In [ ]:
# Test the RAG pipeline
test_query = "What is machine learning?"

result = rag_query(test_query, collection)

print(f"Query: {result['query']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nRetrieved {len(result['retrieved_chunks'])} chunks:")
for i, (chunk, dist) in enumerate(zip(result['retrieved_chunks'], result['distances'])):
    print(f"\n--- Chunk {i+1} (distance: {dist:.4f}) ---")
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)

In [ ]:
# Test with more questions
test_queries = [
    "How does deep learning work?",
    "What are the types of machine learning?",
    "What is natural language processing?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    result = rag_query(query, collection)
    print(f"Q: {result['query']}")
    print(f"A: {result['answer'][:200]}...")

## Summary

In this notebook, we learned:

1. **RAG Architecture**: Combining retrieval and generation
2. **Chunking Strategies**: Fixed, recursive, and semantic chunking
3. **Gemini Embeddings**: Converting text to vector representations
4. **ChromaDB**: Storing and searching embeddings
5. **Complete RAG Pipeline**: From query to answer

## Next Steps
- Try different chunk sizes and overlap values
- Experiment with different document types
- Move to Session 2 for hybrid search!